# Codegen MiniMSMARCO (lexical reranker)

This notebook recreates `configs/codegen/codegen_minimarco.yml` step by step. We train a code-generated lexical reranker over MiniMSMARCO passages, starting from a BM25-style baseline over raw corpus statistics. The dataset only has descriptions (no titles), so the reranker focuses on passage-level signals.

Config values:
- model = gpt-5.5
- reasoning = low
- refresh_every = 3
- search_tools = [get_corpus (raw), fielded_bm25]
- edit.guards = [validation, length(max_lines=10, max_cols=120)]
- eval.train_fraction = 0.20
- eval.seed = 1234
- eval.eval_margin = 0.002
- run.top_k = 10

In [1]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@7273adaab42
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

  Cloning https://github.com/softwaredoug/cheat-at-search.git (to revision 7273adaab42) to /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-lkw669ai
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-lkw669ai
  Running command git checkout -q 7273adaab42
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 7273adaab42
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
2026-05-13 13:31:49,061 - cheat_at_search.data_dir - ERROR - Google Colab drive module not found. Ensure you're running this in Google Colab.


## Get an OpenAI Key + load MiniMSMARCO

This prompts for an OpenAI key so we can run GPT-5.5 during training, then loads the MiniMSMARCO corpus and judgments.

In [2]:
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.minimarco_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

corpus = corpus.reset_index(drop=True)
base_queries = judgments["query"].dropna().unique().tolist()

corpus[["doc_id", "description"]].head(3)

File /Users/doug/.search-experiments/cheat-at-search/msmarco/collectionandqueries.tar.gz already exists, skipping download.
File /Users/doug/.search-experiments/cheat-at-search/msmarco/collectionandqueries.tar.gz already exists, skipping download.


,doc_id,description
0,945864,Notice how their voice goes up and down very d...
1,2776470,Definition of inter-. 1 1 : between : among :...
2,1897608,"Literary consonance. From Wikipedia, the free ..."


## Build search tools (fielded BM25 + raw corpus)

`fielded_bm25` is an agent-callable tool. `get_corpus` is a raw tool that is *only* injected into generated code; we do not pass it to the agent directly.

In [3]:
from typing import Union

from searcharray import SearchArray
from searcharray.similarity import bm25_similarity
from typing_extensions import Literal

from cheat_at_search.tokenizers import snowball_tokenizer

def _parse_weighted_fields(fields: list[str]) -> list[tuple[str, float]]:
    parsed_fields = []
    for field_entry in fields:
        if not isinstance(field_entry, str) or "^" not in field_entry:
            raise ValueError("Fields must be strings in the form 'title^9.3'.")
        field_name, weight_str = field_entry.split("^", 1)
        field_name = field_name.strip()
        if field_name not in {"title", "description"}:
            raise ValueError("Fields must be title or description.")
        try:
            weight = float(weight_str)
        except ValueError as exc:
            raise ValueError("Field weights must be numeric.") from exc
        parsed_fields.append((field_name, weight))
    if not parsed_fields:
        raise ValueError("Fields must include at least one entry.")
    return parsed_fields

def _ensure_field_indices(corpus, field_names: list[str]) -> None:
    for field_name in field_names:
        if field_name not in corpus:
            raise ValueError(f"Corpus missing required field: {field_name}")
        snowball_name = f"{field_name}_snowball"
        if snowball_name not in corpus:
            corpus[snowball_name] = SearchArray.index(corpus[field_name], snowball_tokenizer)

def fielded_bm25(
    keywords: str,
    fields: list[str],
    operator: Literal["and", "or", "phrase"] = "or",
    top_k: int = 5,
    k1: float = 1.2,
    b: float = 0.75,
    agent_state=None,
) -> list[dict[str, Union[str, int, float]]]:
    """Search a corpus using BM25 over weighted fields.

    Args:
        keywords: The search query string.
        fields: Weighted fields to search, e.g. ["title^9.3", "description^4.1"].
        operator: How to combine search terms: and/or/phrase. AND requires every
            term to appear in at least one field. PHRASE treats the entire query
            as a phrase and scores the token list as a single term.
        top_k: The number of top results to return (max 100).
        k1: BM25 k1 parameter.
        b: BM25 b parameter.

    Returns:
        Search results as a list of dictionaries with 'id', 'title',
        'description', and 'score' keys.
    """
    if top_k > 100:
        raise ValueError("top_k must be <= 100")
    if not isinstance(fields, list):
        raise ValueError("fields must be a list of weighted field strings.")
    parsed_fields = _parse_weighted_fields(fields)
    _ensure_field_indices(corpus, [field for field, _ in parsed_fields])

    query_tokens = snowball_tokenizer(keywords)
    scores = np.zeros(len(corpus))
    similarity = bm25_similarity(k1=k1, b=b)
    require_mask = None

    if operator == "phrase":
        if not query_tokens:
            return []
        field_scores = []
        for field_name, weight in parsed_fields:
            snowball_name = f"{field_name}_snowball"
            term_match = corpus[snowball_name].array.score(query_tokens, similarity=similarity)
            field_scores.append(term_match * weight)
        scores += sum(field_scores) if field_scores else 0
    else:
        for token in query_tokens:
            field_scores = []
            for field_name, weight in parsed_fields:
                snowball_name = f"{field_name}_snowball"
                term_match = corpus[snowball_name].array.score(token, similarity=similarity)
                field_scores.append(term_match * weight)
            term_scores = sum(field_scores) if field_scores else 0
            scores += term_scores
            if operator == "and":
                term_present = sum(field_scores) > 0
                require_mask = term_present if require_mask is None else (require_mask & term_present)

        if operator == "and":
            if require_mask is not None:
                scores = scores * require_mask
        elif operator != "or":
            raise ValueError("operator must be 'and', 'or', or 'phrase'")

    top_k_indices = np.argsort(scores)[-top_k:][::-1]
    scores = scores[top_k_indices]
    top_rows = corpus.iloc[top_k_indices].copy()
    top_rows.loc[:, "score"] = scores

    results = []
    for _, row in top_rows.iterrows():
        results.append(
            {
                "id": row.get("doc_id"),
                "title": row.get("title", ""),
                "description": row.get("description", ""),
                "score": row.get("score", 0.0),
            }
        )
    return results

def get_corpus(agent_state=None):
    """Return the pandas DataFrame corpus.

    Columns:
    - title: title (if any) of the document
    - description: body of the document
    - title_snowball: snowball-tokenized SearchArray index for title
    - description_snowball: snowball-tokenized SearchArray index for description

    SearchArray API (lexical statistics) lives on the .array attribute of
    snowball columns, e.g. corpus["title_snowball"].array:

    ## Tokenizing terms + phrases

    Everything below assumes tokenized terms. The tokenizer can be accessed
    through corpus["description_snowball"].array.tokenizer

    As the filed implies _snowball will snowball tokenize:

    tokenizer("tokenized REd apples") -> ["token", "red", "appl"]

    In search array, search with terms by passing a tokens
    Search with phrases by passing a list of tokens

    ## Scoring functions

    ### BM25:

    - score(token): BM25 score of term
    - score([token1, token2, ...], similarity=...): BM25 score of phrase "token1 token2 ..."

    ### Direct tf stats
    - termfreqs(token): term frequency per doc
    - termfreqs([token1, token2, ...]): term frequencies for phrase "token1 token2 ..." per doc

    ### Direct df stats
    - docfreq(token): document frequency

    ### Other
    - doclengths(): document lengths
    """
    return corpus

get_corpus.__name__ = "get_corpus"

tool_fns = [fielded_bm25, get_corpus]

fielded_bm25(base_queries[0], fields=["description^1.0"], operator="or", top_k=3)

[{'id': 1154645,
  'title': '',
  'description': 'Orthopedic physician assistantâ\x80\x99s salary in United States (according to salary experts) ranges from $90,000 to 137,000. The top paying place in the US based on hourly pay is Los Angeles, California where orthopedic physician assistant receives around $63 pay per hour.he hourly pay for orthopedic physician assistant ranges from $40 to $62. Orthopedic physician assistantâ\x80\x99s salary increase with year of experience of orthopedic PA. It also varies from state to state and from city to city.',
  'score': 11.257610827684402},
 {'id': 8505037,
  'title': '',
  'description': 'From millions of real job salary data. 22 Hematology/oncology Physician salary data. Average Hematology/oncology Physician salary is $279,688 Detailed Hematology/oncology Physician starting salary, median salary, pay scale, bonus data report Register & Know how much $ you can earn | Sign In',
  'score': 10.670302212238312},
 {'id': 5072930,
  'title': '',
  '

## Start code from the config

We write the exact `start_code` from the YAML to disk. The agent edits this file with `apply_patch`, and all evaluations read from the file.

In [4]:
from pathlib import Path

START_CODE = '''import numpy as np
import pandas as pd


def rerank_minimarco(query, fielded_bm25, get_corpus, **kwargs):
    corpus = get_corpus()
    snowball = corpus["description_snowball"].array
    tokenizer = snowball.tokenizer
    terms = [term for term in tokenizer(query) if term]
    if not terms:
        return []

    doc_lengths = snowball.doclengths()
    if len(doc_lengths) == 0:
        return []
    avg_dl = float(doc_lengths.mean())
    if avg_dl <= 0:
        return []

    k1 = 0.6
    b = 0.62
    n_docs = len(corpus)
    scores = np.zeros(n_docs)

    for term in terms:
        term_freqs = snowball.termfreqs(term)
        doc_freq = snowball.docfreq(term)
        if doc_freq == 0:
            continue
        idf = np.log(1.0 + (n_docs - doc_freq + 0.5) / (doc_freq + 0.5))
        denom = term_freqs + k1 * (1.0 - b + b * (doc_lengths / avg_dl))
        scores += idf * (term_freqs * (k1 + 1.0)) / np.where(denom == 0, 1.0, denom)

    top_k = int(kwargs.get("top_k", 10))
    if top_k <= 0:
        return []
    ranked = np.argsort(-scores)[:top_k]
    return [str(corpus.iloc[idx]["doc_id"]) for idx in ranked if scores[idx] > 0]
'''

CODE_DIR = Path.cwd() / "codegen_minimarco"
CODE_DIR.mkdir(parents=True, exist_ok=True)
code_path = CODE_DIR / "rerank_minimarco.py"
code_path.write_text(START_CODE, encoding="utf-8")

START_CODE

'import numpy as np\nimport pandas as pd\n\n\ndef rerank_minimarco(query, fielded_bm25, get_corpus, **kwargs):\n    corpus = get_corpus()\n    snowball = corpus["description_snowball"].array\n    tokenizer = snowball.tokenizer\n    terms = [term for term in tokenizer(query) if term]\n    if not terms:\n        return []\n\n    doc_lengths = snowball.doclengths()\n    if len(doc_lengths) == 0:\n        return []\n    avg_dl = float(doc_lengths.mean())\n    if avg_dl <= 0:\n        return []\n\n    k1 = 0.6\n    b = 0.62\n    n_docs = len(corpus)\n    scores = np.zeros(n_docs)\n\n    for term in terms:\n        term_freqs = snowball.termfreqs(term)\n        doc_freq = snowball.docfreq(term)\n        if doc_freq == 0:\n            continue\n        idf = np.log(1.0 + (n_docs - doc_freq + 0.5) / (doc_freq + 0.5))\n        denom = term_freqs + k1 * (1.0 - b + b * (doc_lengths / avg_dl))\n        scores += idf * (term_freqs * (k1 + 1.0)) / np.where(denom == 0, 1.0, denom)\n\n    top_k = int(

## Training split, guardrails, and eval helpers

We split queries into train/validation (20% train, remainder validation), set the eval margin to 0.002, and create guardrails. We also build `run_reranker`/`run_evals` helpers so the agent can inspect progress.

In [5]:
import inspect
import logging
import random

from cheat_at_search.agent.openai_agent import OpenAIAgent
from cheat_at_search.search import run_strategy
from cheat_at_search.strategy.strategy import SearchStrategy
from cheat_at_search.tools.code import make_length_validator, make_patch_fn, make_run_path_grep_tool
from cheat_at_search.tools.eval import CodeGenSearchStrategy

TRAIN_FRACTION = 0.20
SEED = 1234
EVAL_MARGIN = 0.002
REFRESH_EVERY = 3
WORKERS = 1

logger = logging.getLogger("codegen_minimarco")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

def split_queries(queries, train_size, seed):
    if not queries:
        return [], []
    train_size = min(max(train_size, 1), len(queries))
    shuffled = list(queries)
    random.Random(seed).shuffle(shuffled)
    return shuffled[:train_size], shuffled[train_size:]

def load_rerank_fn(code, name):
    namespace = {}
    exec(code, namespace, namespace)
    fn = namespace.get(name)
    if not callable(fn):
        raise ValueError(f"Missing rerank function: {name}")
    return fn

def make_codegen_strategy(code_text):
    params = inspect.signature(CodeGenSearchStrategy).parameters
    if "tool_fns" in params:
        kwargs = {}
        if "search_fn" in params:
            kwargs["search_fn"] = fielded_bm25
        kwargs["tool_fns"] = tool_fns
        if "code" in params:
            kwargs["code"] = code_text
        if "code_path" in params:
            kwargs["code_path"] = code_path
        if "rerank_name" in params:
            kwargs["rerank_name"] = "rerank_minimarco"
        if "workers" in params:
            kwargs["workers"] = WORKERS
        return CodeGenSearchStrategy(corpus, **kwargs)

    class NotebookCodeGenStrategy(SearchStrategy):
        def __init__(self, corpus, tool_fns, code, rerank_name, workers):
            self.corpus = corpus
            self.tool_fns = tool_fns
            self.code = code
            self.rerank_name = rerank_name
            self.workers = workers
            self.name = "notebook_codegen"
            if "doc_id" in corpus.columns:
                self.id_lookup = {str(doc_id): idx for idx, doc_id in enumerate(corpus["doc_id"])}
            else:
                self.id_lookup = {str(idx): idx for idx in range(len(corpus))}

        def search(self, query, k=10):
            rerank_fn = load_rerank_fn(self.code, self.rerank_name)
            doc_ids = rerank_fn(query, *self.tool_fns, top_k=k)
            scores = np.arange(len(doc_ids), 0, -1)
            top_k_indices = []
            top_scores = []
            for doc_id, score in zip(doc_ids, scores):
                idx = self.id_lookup.get(str(doc_id))
                if idx is None:
                    continue
                top_k_indices.append(idx)
                top_scores.append(float(score))
            return top_k_indices, top_scores

    return NotebookCodeGenStrategy(corpus, tool_fns, code_text, "rerank_minimarco", WORKERS)

def make_eval_guardrail(*, corpus, judgments, tool_fns, rerank_name, seed, num_queries, queries=None, workers=1):
    def eval_guardrail(code: str):
        strategy = make_codegen_strategy(code)
        results = run_strategy(
            strategy,
            judgments,
            queries=queries,
            num_queries=None if queries else num_queries,
            seed=seed,
            cache=False,
        )
        ndcgs = results.groupby("query")["ndcg"].mean()
        return ndcgs

    return eval_guardrail

def make_eval_tools(code_path, tool_fns, seed, num_queries, queries):
    def run_evals():
        """Evaluate the reranker on a sample of queries."""
        code = code_path.read_text(encoding="utf-8")
        strategy = make_codegen_strategy(code)
        results = run_strategy(
            strategy,
            judgments,
            queries=queries,
            num_queries=None if queries else num_queries,
            seed=seed,
            cache=False,
        )
        ndcgs = results.groupby("query")["ndcg"].mean()
        return {
            "mean_ndcg": float(ndcgs.mean()) if not ndcgs.empty else 0.0,
            "query_ndcgs": {str(k): float(v) for k, v in ndcgs.to_dict().items()},
        }

    def run_reranker(query, label=False):
        """Run the reranker on a query.

        Set label=True to include human labels if available.
        """
        code = code_path.read_text(encoding="utf-8")
        rerank_fn = load_rerank_fn(code, "rerank_minimarco")
        doc_ids = rerank_fn(query, *tool_fns)
        scores = np.arange(len(doc_ids), 0, -1)
        results = []
        label_map = None
        if label and "query" in judgments.columns and "doc_id" in judgments.columns:
            subset = judgments[judgments["query"] == query]
            if not subset.empty:
                label_map = dict(zip(subset["doc_id"], subset.get("grade", [])))
        for doc_id, score in zip(doc_ids, scores):
            row = corpus[corpus["doc_id"] == doc_id]
            if row.empty:
                continue
            row = row.iloc[0]
            entry = {
                "doc_id": str(row.get("doc_id")),
                "title": row.get("title", ""),
                "description": row.get("description", ""),
                "score": int(score),
            }
            if label_map is not None:
                entry["grade"] = label_map.get(row.get("doc_id"))
            results.append(entry)
        return results

    return run_evals, run_reranker

train_size = int(len(base_queries) * TRAIN_FRACTION)
training_queries, validation_queries = split_queries(base_queries, train_size, SEED)

training_eval_fn = make_eval_guardrail(
    corpus=corpus,
    judgments=judgments,
    tool_fns=tool_fns,
    rerank_name="rerank_minimarco",
    seed=SEED,
    num_queries=len(training_queries),
    queries=training_queries,
    workers=WORKERS,
)
validation_eval_fn = None
if validation_queries:
    validation_eval_fn = make_eval_guardrail(
        corpus=corpus,
        judgments=judgments,
        tool_fns=tool_fns,
        rerank_name="rerank_minimarco",
        seed=SEED,
        num_queries=len(validation_queries),
        queries=validation_queries,
        workers=WORKERS,
    )

length_guard = make_length_validator(max_lines=10, max_cols=120)
run_evals, run_reranker = make_eval_tools(code_path, tool_fns, SEED, len(training_queries), training_queries)

## Single training run (one agent iteration)

We create patching tools, inject the raw `get_corpus` description into the prompt, and let the agent attempt a single improvement.

In [11]:
from pydantic import BaseModel, Field

apply_patch, try_out_patch, revert_changes = make_patch_fn(
    search_fn=fielded_bm25,
    corpus=corpus,
    code_dir=str(CODE_DIR),
    tool_fns=tool_fns,
    module_name=code_path.stem,
    function_name="rerank_minimarco",
    guardrail_fns=[length_guard],
    training_eval_fn=training_eval_fn,
    validation_eval_fn=validation_eval_fn,
    eval_margin=EVAL_MARGIN,
    logger=logger,
)

grep_run_path = make_run_path_grep_tool(
    Path("runs/codegen/minimarco/codegen_minimarco/20260504_033459"),
    logger=logger,
)

SYSTEM_PROMPT = '''Your task is to improve the reranker code so it returns more relevant results.

Use apply_patch to edit the reranker module.
Use run_reranker to inspect single queries and run_evals for NDCG.
Use grep to explore past runs in a path holding previous codegen experiments for ideas / what to avoid.

If NDCG does not improve, revert with revert_changes.

You start with BM25 on raw corpus stats. But try to improve on it. The goal: find a better
lexical retrieval function. 

This is a passage dataset ONLY has description. No _title_. Find a better passage ranker.

DO NOT rename the function in the code. You MUST keep the signature the same.

HINTS - what to explore
- Find statistically significant bigrams (colocations) using the phrase term frequency
  stats
- Use the fielded BM25 scores as a signal in your reranker
- Try reranking on exact phrase matches in the retrieved candidates
- Use term length as another measure of specificity (ie longer terms are more specific to user intent)
- Try different stopword removal opproaches (but don't apply it to short queries)
- Look in logs where there was ALMOST a 0.002 improvement and see what the code changes were. Try to understand why they were close but not quite there - maybe you can mitigate the downsides to get a net improvement?
'''

raw_tool_doc = get_corpus.__doc__ or "Return the corpus DataFrame."
SYSTEM_PROMPT += (
    "\n\n## Additionally injected search code\n\n"
    "The following functions are available to generated code, injected into the rerank function:\n\n"
    "### get_corpus\n\n"
    f"{raw_tool_doc}"
)

prompt = SYSTEM_PROMPT + f"\n\nReranker code to improve:\n\n{code_path.read_text(encoding='utf-8')}\n"

class FinalMessage(BaseModel):
    """Final message indicating completion of the reranker improvement process."""
    message: str = Field(..., description="A message indicating that the reranker improvement process is complete.")

tools = [
    fielded_bm25,
    apply_patch,
    run_reranker,
    run_evals,
    grep_run_path,
    revert_changes,
]

def make_agent_kwargs(prompt_text):
    params = inspect.signature(OpenAIAgent).parameters
    kwargs = {}
    if "tools" in params:
        kwargs["tools"] = tools
    if "model" in params:
        kwargs["model"] = "openai/gpt-5-mini"
    if "system_prompt" in params:
        kwargs["system_prompt"] = prompt_text
    elif "prompt" in params:
        kwargs["prompt"] = prompt_text
    if "response_model" in params:
        kwargs["response_model"] = FinalMessage
    if "reasoning_level" in params:
        kwargs["reasoning_level"] = "low"
    return kwargs

agent = OpenAIAgent(**make_agent_kwargs(prompt))
agent_inputs = [{"role": "user", "content": prompt}]

try:
    agent.loop(inputs=agent_inputs)
except Exception as exc:
    logger.error("Agent loop failed: %s", exc)

2026-05-13 14:19:52,388 INFO Applying patch with edits
INFO:codegen_minimarco:Applying patch with edits
2026-05-13 14:19:52,389 INFO Patching code with edits
INFO:codegen_minimarco:Patching code with edits
2026-05-13 14:19:52,389 INFO Goal: Improve lexical reranker by adding term specificity, phrase boosts, and fielded BM25 fusion.
INFO:codegen_minimarco:Goal: Improve lexical reranker by adding term specificity, phrase boosts, and fielded BM25 fusion.
2026-05-13 14:19:52,390 INFO Why: Longer terms are more specific, bigram and full-phrase matches signal relevance, and using fielded BM25 score adds complementary signal.
INFO:codegen_minimarco:Why: Longer terms are more specific, bigram and full-phrase matches signal relevance, and using fielded BM25 score adds complementary signal.
2026-05-13 14:19:52,390 INFO Expected improved queries: ['how to fix a torn screen', 'what causes slow internet connection', 'best way to cook brown rice']
INFO:codegen_minimarco:Expected improved queries: ['

## Training loop (refresh every 3 rounds)

Now we wrap the same process in a loop. Every three rounds we resample the training/validation queries and rebuild eval tools.

In [ ]:
ROUNDS = 3

for round_idx in range(ROUNDS):
    if round_idx % REFRESH_EVERY == 0:
        seed = SEED + round_idx
        training_queries, validation_queries = split_queries(base_queries, train_size, seed)
        training_eval_fn = make_eval_guardrail(
            corpus=corpus,
            judgments=judgments,
            tool_fns=tool_fns,
            rerank_name="rerank_minimarco",
            seed=seed,
            num_queries=len(training_queries),
            queries=training_queries,
            workers=WORKERS,
        )
        validation_eval_fn = None
        if validation_queries:
            validation_eval_fn = make_eval_guardrail(
                corpus=corpus,
                judgments=judgments,
                tool_fns=tool_fns,
                rerank_name="rerank_minimarco",
                seed=seed,
                num_queries=len(validation_queries),
                queries=validation_queries,
                workers=WORKERS,
            )
        run_evals, run_reranker = make_eval_tools(code_path, tool_fns, seed, len(training_queries), training_queries)

    apply_patch, try_out_patch, revert_changes = make_patch_fn(
        search_fn=fielded_bm25,
        corpus=corpus,
        code_dir=str(CODE_DIR),
        tool_fns=tool_fns,
        module_name=code_path.stem,
        function_name="rerank_minimarco",
        guardrail_fns=[length_guard],
        training_eval_fn=training_eval_fn,
        validation_eval_fn=validation_eval_fn,
        eval_margin=EVAL_MARGIN,
        logger=logger,
    )

    tools = [
        fielded_bm25,
        apply_patch,
        run_reranker,
        run_evals,
        grep_run_path,
        revert_changes,
    ]

    agent = OpenAIAgent(**make_agent_kwargs(prompt))
    agent_inputs = [{"role": "user", "content": prompt}]

    logger.info("Starting round %s", round_idx + 1)
    try:
        agent.loop(inputs=agent_inputs)
    except Exception as exc:
        logger.error("Agent loop failed: %s", exc)

2026-05-13 14:21:55,253 INFO Starting round 1
INFO:codegen_minimarco:Starting round 1
2026-05-13 14:22:19,144 INFO Applying patch with edits
INFO:codegen_minimarco:Applying patch with edits
2026-05-13 14:22:19,146 INFO Patching code with edits
INFO:codegen_minimarco:Patching code with edits
2026-05-13 14:22:19,147 INFO Goal: Improve lexical reranker by adding term length weighting, stronger phrase and bigram boosts, tightness normalization, and incorporating fielded BM25 as auxiliary signal.
INFO:codegen_minimarco:Goal: Improve lexical reranker by adding term length weighting, stronger phrase and bigram boosts, tightness normalization, and incorporating fielded BM25 as auxiliary signal.
2026-05-13 14:22:19,149 INFO Why: These changes capture phrase collocations, term specificity, and leverage existing BM25 over the description field to improve ranking quality without changing the function signature.
INFO:codegen_minimarco:Why: These changes capture phrase collocations, term specificity

## Run the trained strategy

After training, we load the latest reranker code and evaluate it over MiniMSMARCO.

In [10]:
from cheat_at_search.search import ndcgs, mrrs

trained_code = code_path.read_text(encoding="utf-8")
strategy = make_codegen_strategy(trained_code)
results = run_strategy(strategy, judgments, cache=False)

{
    "ndcg_mean": float(ndcgs(results).mean()),
    "mrr_mean": float(mrrs(results).mean()),
}

Searching: 100%|████████████████████████████████████████████████████████████| 543/543 [00:30<00:00, 17.70it/s]


{'ndcg_mean': 0.17625888779687432, 'mrr_mean': 0.5162566868367974}